# Education per municipality

This notebook processes the CBS education data for Dutch municipalities (2024) and saves the result for the other notebooks.

**Input file (`data/raw/`):**
- `kwb2024_per_gemeente.csv`: counts of residents by education level per municipality.

**Output (`data/processed/`):**
- `education_per_municipality.csv`: one row per municipality (342), with percentages of highly and low educated residents.


In [1]:
import pandas as pd
from pathlib import Path


In [2]:
# Paths relative to the notebooks folder
raw_path = Path("../data/raw/kwb2024_per_gemeente.csv")
output_path = Path("../data/processed/education_per_municipality.csv")

# Load the data
education = pd.read_csv(raw_path, encoding="utf-8-sig")

# Select the relevant columns
education = education[["gemeente", "opl_laag", "opl_middel", "opl_hoog"]].copy()

# Calculate the percentages of low and highly educated residents
education_total = education[["opl_laag", "opl_middel", "opl_hoog"]].sum(axis=1, min_count=3)
education["highly_educated_pct"] = (100 * education["opl_hoog"] / education_total).round(2)
education["low_educated_pct"] = (100 * education["opl_laag"] / education_total).round(2)

# Align municipality names with the other processed datasets
name_mapping = {
    "'s-Gravenhage": "Den Haag",
    "Beek (L.)": "Beek",
    "Hengelo (O.)": "Hengelo",
    "Rijswijk (ZH.)": "Rijswijk",
    "Stein (L.)": "Stein",
    "Middelburg (Z.)": "Middelburg",
    "Laren (NH.)": "Laren",
}
education["gemeente"] = education["gemeente"].replace(name_mapping)

# Keep the required output columns and their exact names
data = education.rename(columns={"gemeente": "municipality"})[
    ["municipality", "highly_educated_pct", "low_educated_pct"]
]

# Check output shape and municipality uniqueness
assert data.shape == (342, 3), f"Unexpected shape: {data.shape}"
assert data["municipality"].is_unique, "Duplicate municipalities found"
print(data.shape)
print(data.isna().sum())

# Save with the existing filename
output_path.parent.mkdir(parents=True, exist_ok=True)
data.to_csv(output_path, index=False)


(342, 3)
municipality           0
highly_educated_pct    1
low_educated_pct       1
dtype: int64


In [3]:
data.head()


,municipality,highly_educated_pct,low_educated_pct
0,Groningen,44.74,15.90
1,Almere,30.76,26.89
2,Stadskanaal,17.49,31.98
3,Veendam,20.85,30.83
4,Zeewolde,31.56,22.51
